In [190]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,roc_auc_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier



In [191]:
df=pd.read_csv("../data/processed/customer_churn_cleaned3.csv")

In [192]:
bins = [300, 580, 670, 740, 800, 900]
labels = ["Poor", "Fair", "Good", "Very Good", "Excellent"]

df["CreditCategory"] = pd.cut(df["CreditScore"], bins=bins, labels=labels)

In [193]:
df = pd.get_dummies(df, columns=["CreditCategory"], drop_first=False)

In [194]:
X = df.drop(["Exited","Age"], axis=1)
y = df["Exited"]

In [195]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CreditScore               10000 non-null  int64  
 1   Gender                    10000 non-null  int64  
 2   Tenure                    10000 non-null  int64  
 3   Balance                   10000 non-null  float64
 4   NumOfProducts             10000 non-null  int64  
 5   HasCrCard                 10000 non-null  int64  
 6   IsActiveMember            10000 non-null  int64  
 7   EstimatedSalary           10000 non-null  float64
 8   Geography_France          10000 non-null  int64  
 9   Geography_Germany         10000 non-null  int64  
 10  Geography_Spain           10000 non-null  int64  
 11  AgeGroup_18-30            10000 non-null  bool   
 12  AgeGroup_31-40            10000 non-null  bool   
 13  AgeGroup_41-50            10000 non-null  bool   
 14  AgeGrou

In [196]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [197]:
scaler = StandardScaler()

numerical_cols = [
    "CreditScore",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "EstimatedSalary"
]
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [198]:
X_train["BalancePerProduct"] = X_train["Balance"] / X_train["NumOfProducts"].replace(0, 1)
X_test["BalancePerProduct"] = X_test["Balance"] / X_test["NumOfProducts"].replace(0, 1)

In [199]:
# X_train.drop(['Balance', 'NumOfProducts'], axis=1, inplace=True)
# X_test.drop(['Balance', 'NumOfProducts'], axis=1, inplace=True)

In [200]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [201]:

def objective(trial):

    params = {
        "C": trial.suggest_float("C", 1e-3, 100, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 1, log=True),
        "kernel": "rbf",
        "class_weight": "balanced"
    }

    model = SVC(
        **params,
        probability=True
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    ).mean()

    return score

In [202]:

# Create the study
study = optuna.create_study(direction="maximize")

# Run optimization
study.optimize(objective, n_trials=50)

# Best hyperparameters
print("Best Parameters:")
print(study.best_params)

print("\nBest CV ROC-AUC:")
print(study.best_value)

# Train the final model with the best parameters
best_svm = SVC(
    C=study.best_params["C"],
    gamma=study.best_params["gamma"],
    kernel="rbf",
    class_weight="balanced",
    probability=True,
    random_state=42
)

best_svm.fit(X_train, y_train)

# Predictions
y_train_pred = best_svm.predict(X_train)
y_test_pred = best_svm.predict(X_test)

# Probabilities for ROC-AUC
y_prob = best_svm.predict_proba(X_test)[:, 1]

print("\nBest SVM Model Evaluation:")
# Evaluation
print("\nTrain Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

[I 2026-07-10 04:10:39,117] A new study created in memory with name: no-name-fe5d6ad2-a323-4c51-b3b8-f26e9bc18101
[I 2026-07-10 04:11:55,752] Trial 0 finished with value: 0.5607708680451887 and parameters: {'C': 0.022479738804586796, 'gamma': 0.00045455941274468776}. Best is trial 0 with value: 0.5607708680451887.
[I 2026-07-10 04:13:11,815] Trial 1 finished with value: 0.5054896899769818 and parameters: {'C': 0.006926758051557108, 'gamma': 0.07019359833909554}. Best is trial 0 with value: 0.5607708680451887.
[I 2026-07-10 04:14:15,004] Trial 2 finished with value: 0.5331832497038457 and parameters: {'C': 0.05625510900389151, 'gamma': 0.004464549344668559}. Best is trial 0 with value: 0.5607708680451887.
[I 2026-07-10 04:15:19,439] Trial 3 finished with value: 0.5184720844449153 and parameters: {'C': 0.0059441188043258065, 'gamma': 0.012474945762658073}. Best is trial 0 with value: 0.5607708680451887.
[I 2026-07-10 04:16:27,826] Trial 4 finished with value: 0.5599722144638883 and param

KeyboardInterrupt: 

In [203]:
y_pred = model.predict(X_test_scaled)
y_pred_train = model.predict(X_train_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Train Accuracy:", accuracy_score(y_train, y_pred_train))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC AUC:", roc_auc)

Accuracy: 0.8285
Train Accuracy: 0.82425

Confusion Matrix:
[[1542   51]
 [ 292  115]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.97      0.90      1593
           1       0.69      0.28      0.40       407

    accuracy                           0.83      2000
   macro avg       0.77      0.63      0.65      2000
weighted avg       0.81      0.83      0.80      2000

ROC AUC: 0.7940930144319974


Better with (balance , products) and categorize the credit score

In [204]:
svm = SVC(
    kernel='rbf',
    random_state=42,
    probability=True
)

svm.fit(X_train_scaled, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [205]:
y_pred_svm = svm.predict(X_test_scaled)
y_pred_svm_train=svm.predict(X_train_scaled)
scores = svm.decision_function(X_test_scaled)
roc_auc = roc_auc_score(y_test, scores)
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("ROC AUC:", roc_auc)

Accuracy: 0.8585
ROC AUC: 0.8347068177576652


In [206]:
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))
print(confusion_matrix(y_test, y_pred_svm))
print("Train Accuracy:", accuracy_score(y_train, y_pred_svm_train))


Accuracy: 0.8585
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.79      0.41      0.54       407

    accuracy                           0.86      2000
   macro avg       0.83      0.69      0.73      2000
weighted avg       0.85      0.86      0.84      2000

[[1549   44]
 [ 239  168]]
Train Accuracy: 0.8645


In [207]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [208]:
y_pred_xgb = xgb.predict(X_test)
y_pred_xgb_train = xgb.predict(X_train)
roc_auc_xgb = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Train Accuracy:", accuracy_score(y_train, y_pred_xgb_train))
print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", roc_auc_xgb)

Accuracy: 0.8465
Train Accuracy: 0.95925
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      1593
           1       0.69      0.45      0.54       407

    accuracy                           0.85      2000
   macro avg       0.78      0.70      0.73      2000
weighted avg       0.83      0.85      0.83      2000

ROC AUC: 0.8312364753042719


In [209]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight="balanced",  # Helpful for imbalanced datasets
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_pred_rf_train = rf.predict(X_train)   
print("Random Forest Classifier Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Train Accuracy:", accuracy_score(y_train, y_pred_rf_train))
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

Random Forest Classifier Results:
Accuracy: 0.856
Train Accuracy: 1.0
              precision    recall  f1-score   support

           0       0.87      0.97      0.91      1593
           1       0.77      0.42      0.54       407

    accuracy                           0.86      2000
   macro avg       0.82      0.69      0.73      2000
weighted avg       0.85      0.86      0.84      2000

ROC AUC: 0.8418881130745538


In [210]:

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)
print(importance.head(15))

              Feature  Importance
7     EstimatedSalary    0.126720
0         CreditScore    0.119749
4       NumOfProducts    0.115740
21  BalancePerProduct    0.112765
3             Balance    0.108710
2              Tenure    0.081755
14     AgeGroup_51-60    0.044436
6      IsActiveMember    0.037613
12     AgeGroup_31-40    0.036070
11     AgeGroup_18-30    0.033228
13     AgeGroup_41-50    0.031006
9   Geography_Germany    0.025033
1              Gender    0.023804
5           HasCrCard    0.020092
8    Geography_France    0.014819
